# NB_patient_360

Builds `adb_caresync_preprod_preprod.gold.patient_360` from the silver patient, hospital, insurance, and lab result tables defined in the STTM workbook, then validates the output schema and sample rows.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresync_preprod.gold;

CREATE OR REPLACE TABLE adb_caresync_preprod.gold.patient_360 AS
WITH lab_results_agg AS (
  SELECT
    patient_id,
    CAST(COUNT(*) AS INT) AS total_tests,
    CAST(SUM(CASE WHEN result_status = 'Abnormal' THEN 1 ELSE 0 END) AS INT) AS abnormal_test_count,
    CAST(SUM(CASE WHEN result_status = 'Critical' THEN 1 ELSE 0 END) AS INT) AS critical_test_count,
    MAX(test_date) AS last_test_date
  FROM adb_caresync_preprod.silver.lab_results
  GROUP BY patient_id
)
SELECT
  p.patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE) AS date_of_birth,
  CAST(FLOOR(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  p.registered_hospital_id AS hospital_id,
  h.hospital_name,
  p.insurance_provider_id,
  ip.provider_name AS insurance_provider_name,
  COALESCE(lra.total_tests, 0) AS total_tests,
  COALESCE(lra.abnormal_test_count, 0) AS abnormal_test_count,
  COALESCE(lra.critical_test_count, 0) AS critical_test_count,
  lra.last_test_date
FROM adb_caresync_preprod.silver.patients AS p
LEFT JOIN adb_caresync_preprod.silver.hospitals AS h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_caresync_preprod.silver.insurance_providers AS ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_results_agg AS lra
  ON p.patient_id = lra.patient_id;

num_affected_rows,num_inserted_rows


In [0]:
updated_df = spark.table("adb_caresync_preprod.gold.patient_360")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))

In [0]:
%sql
Select * from adb_caresync_preprod.gold.patient_360 limit 10

patient_id,first_name,last_name,date_of_birth,age,gender,city,state,hospital_id,hospital_name,insurance_provider_id,insurance_provider_name,total_tests,abnormal_test_count,critical_test_count,last_test_date
1,Mary,Jones,2024-02-15,2,M,Austin,TX,1,Austin General Hospital,5,Medicare Government Plan,2,0,0,2025-03-13
2,Mary,Johnson,1967-06-26,59,F,Dallas,TX,2,Dallas Medical Center,8,Kaiser Permanente Select,1,1,0,2026-01-02
3,Mary,Jackson,1975-05-14,51,M,Houston,TX,3,Houston Regional Hospital,11,TRICARE Military Health,1,1,0,2025-01-04
4,Mary,Thomas,2010-10-05,15,F,San Antonio,TX,4,San Antonio Community Hospital,14,WellCare Advantage,2,0,0,2026-01-26
5,Mary,Gonzalez,2000-08-22,26,M,Fort Worth,TX,5,Fort Worth Health Center,17,EmblemHealth Choice,1,0,0,2025-07-13
6,Mary,Martinez,1954-03-22,72,F,Phoenix,AZ,6,Phoenix Care Center,20,SunLife Regional Health,2,0,0,2025-12-27
7,Mary,Miller,1985-08-18,41,M,Denver,CO,7,Denver Memorial Hospital,23,Baylor Scott Health Plan,1,0,0,2025-07-01
8,Mary,Brown,1951-01-20,75,F,Seattle,WA,8,Seattle General Hospital,1,BlueShield Health,2,0,0,2025-11-24
9,Mary,Smith,1951-03-20,75,M,Portland,OR,9,Portland Medical Center,4,Cigna Health Partners,2,0,0,2025-06-11
10,Mary,Moore,1952-11-14,73,F,Chicago,IL,10,Chicago Regional Hospital,7,Humana Care Plus,1,1,0,2025-03-13
